# PoNHy — Potential for Natural Hydrogen
### A Guided Notebook: Unpacking, Using, and Visualizing the Data

> **PoNHy** is an open-source Python framework for estimating H₂ generation from serpentinization under geological, physical, and chemical conditions. It integrates 3D geophysical models, thermodynamic equilibrium calculations, and parametric simulations of fluid–rock interactions.
>
> GitHub: [RodolfoChristiansen/PoNHy](https://github.com/RodolfoChristiansen/PoNHy)

---

## Notebook Contents
1. **Environment Setup & Repo Clone**
2. **Data Unpacking** — topography, gravity, magnetics, density/susceptibility grids
3. **Thermodynamic Database Exploration** — the Zarr H₂ production databases
4. **Core Physics** — serpentinization degree, H₂ solubility, lithostatic pressure
5. **Production Rate Calculation** — volumetric H₂ estimates from inversion outputs
6. **Monte Carlo Sensitivity** — lightweight no-saturation parameter sweep
7. **Full Visualization Suite** — maps, cross-sections, heatmaps, MC distributions
8. **Interactive Parameter Explorer** — ipywidgets slider dashboard


---
## 1. Environment Setup & Repo Clone

In [ ]:
# Install dependencies (run once)
import subprocess, sys

packages = [
    'numpy', 'pandas', 'matplotlib', 'scipy', 'xarray', 'zarr',
    'pyyaml', 'tqdm', 'ipywidgets'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✓ All packages installed')

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/RodolfoChristiansen/PoNHy.git'
REPO_DIR = 'PoNHy'

if not os.path.isdir(REPO_DIR):
    print('Cloning PoNHy repository...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('✓ Cloned')
else:
    print('✓ Repository already present')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print('Files:', sorted(os.listdir('.')))

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.interpolate import griddata
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

DATA_DIR = 'Data_Pyrenees'
DB_DIR   = os.path.join(DATA_DIR, 'DB0')
print('✓ Imports OK')

---
## 2. Data Unpacking

PoNHy uses four types of input data files:
| File | Format | Description |
|---|---|---|
| `Ext_Topo.txt` | TSV: X Y Z | Topography / receiver elevations (m, UTM) |
| `Ext_Grav.txt` | TSV: X Y Z grav | Bouguer gravity anomaly (mGal) |
| `Ext_Mag_down.txt` | TSV: X Y Z mag | Total magnetic intensity anomaly (nT) |
| `Density_Serpentinite.csv` | CSV: X Y Z Density | Inverted density contrast (g/cc) |
| `Magsus_Serpentinite.csv` | CSV: X Y Z Magsus | Inverted magnetic susceptibility (SI) |
| `temperature_70.csv` | CSV: X Y Z T | 3D temperature field (°C) |

In [ ]:
# --- Load topography ---
topo = np.loadtxt(os.path.join(DATA_DIR, 'Ext_Topo.txt'))
print(f'Topography: {topo.shape[0]} points')
print(f'  X range: {topo[:,0].min():.0f} – {topo[:,0].max():.0f} m')
print(f'  Y range: {topo[:,1].min():.0f} – {topo[:,1].max():.0f} m')
print(f'  Z range: {topo[:,2].min():.1f} – {topo[:,2].max():.1f} m')

In [ ]:
# --- Load gravity and magnetics ---
grav_raw = np.loadtxt(os.path.join(DATA_DIR, 'Ext_Grav.txt'))
mag_raw  = np.loadtxt(os.path.join(DATA_DIR, 'Ext_Mag_down.txt'))

xyz_obs = grav_raw[:, :3]
dobs_grav = grav_raw[:, -1]
dobs_mag  = mag_raw[:, -1]

print(f'Gravity observations: {len(dobs_grav)} points')
print(f'  Range: {dobs_grav.min():.3f} – {dobs_grav.max():.3f} mGal')
print(f'Magnetic observations: {len(dobs_mag)} points')
print(f'  Range: {dobs_mag.min():.1f} – {dobs_mag.max():.1f} nT')

In [ ]:
# --- Load inversion outputs: density & susceptibility ---
def load_xyz_data(path):
    """Load X Y Z value CSV (skip header comment if present)."""
    df = pd.read_csv(path, comment='#', header=None, names=['X','Y','Z','Value'])
    return df

dens_df  = load_xyz_data(os.path.join(DATA_DIR, 'Density_Serpentinite.csv'))
magsus_df = load_xyz_data(os.path.join(DATA_DIR, 'Magsus_Serpentinite.csv'))

dens_valid  = dens_df.dropna()
magsus_valid = magsus_df.dropna()

print(f'Density model:       {len(dens_valid):,} valid cells  (of {len(dens_df):,} total)')
print(f'  Density contrast range: {dens_valid.Value.min():.4f} – {dens_valid.Value.max():.4f} g/cc')
print(f'Susceptibility model: {len(magsus_valid):,} valid cells')
print(f'  Susceptibility range: {magsus_valid.Value.min():.5f} – {magsus_valid.Value.max():.5f} SI')

In [ ]:
# --- Load 3D temperature field ---
temp_df = pd.read_csv(os.path.join(DATA_DIR, 'temperature_70.csv'))
print(f'Temperature field: {len(temp_df):,} points')
print(f'  Columns: {list(temp_df.columns)}')
print(f'  Depth range: {temp_df.Z.min():.1f} – {temp_df.Z.max():.1f} m')
print(f'  T range: {temp_df.Temperature.min():.1f} – {temp_df.Temperature.max():.1f} °C')

In [ ]:
# ── Figure 1: Input Data Maps ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('PoNHy — Input Geophysical Data (Pyrenees Case Study)', fontsize=14, fontweight='bold', y=1.02)

x, y = xyz_obs[:, 0], xyz_obs[:, 1]
xi = np.linspace(x.min(), x.max(), 60)
yi = np.linspace(y.min(), y.max(), 60)
Xi, Yi = np.meshgrid(xi, yi)

# Topography
Zi_topo = griddata((topo[:,0], topo[:,1]), topo[:,2], (Xi, Yi), method='linear')
c0 = axes[0].pcolormesh(Xi/1e3, Yi/1e3, Zi_topo, cmap='terrain', shading='auto')
plt.colorbar(c0, ax=axes[0], label='Elevation (m)')
axes[0].set_title('Topography', fontweight='bold')
axes[0].set_xlabel('Easting (km)'); axes[0].set_ylabel('Northing (km)')

# Gravity
Zi_grav = griddata((x, y), dobs_grav, (Xi, Yi), method='linear')
c1 = axes[1].pcolormesh(Xi/1e3, Yi/1e3, Zi_grav, cmap='RdBu_r', shading='auto')
plt.colorbar(c1, ax=axes[1], label='Gravity anomaly (mGal)')
axes[1].set_title('Bouguer Gravity Anomaly', fontweight='bold')
axes[1].set_xlabel('Easting (km)')

# Magnetics
Zi_mag = griddata((x, y), dobs_mag, (Xi, Yi), method='linear')
c2 = axes[2].pcolormesh(Xi/1e3, Yi/1e3, Zi_mag, cmap='PuOr_r', shading='auto')
plt.colorbar(c2, ax=axes[2], label='Magnetic TMI anomaly (nT)')
axes[2].set_title('Total Magnetic Intensity Anomaly', fontweight='bold')
axes[2].set_xlabel('Easting (km)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Density & Susceptibility Model (map view at shallowest valid depth) ──
# Pick the shallowest depth slice
shallow_z = dens_valid.Z.max()
dens_slice  = dens_valid[dens_valid.Z == shallow_z]
magsus_slice = magsus_valid[magsus_valid.Z == shallow_z]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Inversion Model — Shallow Slice (Z ≈ {shallow_z:.0f} m)', fontsize=13, fontweight='bold')

sc1 = axes[0].scatter(dens_slice.X/1e3, dens_slice.Y/1e3, c=dens_slice.Value,
                      cmap='seismic', s=18, vmin=-0.3, vmax=-0.1)
plt.colorbar(sc1, ax=axes[0], label='Density contrast (g/cc)')
axes[0].set_title('Density Model', fontweight='bold')
axes[0].set_xlabel('Easting (km)'); axes[0].set_ylabel('Northing (km)')

sc2 = axes[1].scatter(magsus_slice.X/1e3, magsus_slice.Y/1e3, c=magsus_slice.Value,
                      cmap='magma', s=18)
plt.colorbar(sc2, ax=axes[1], label='Susceptibility (SI)')
axes[1].set_title('Magnetic Susceptibility Model', fontweight='bold')
axes[1].set_xlabel('Easting (km)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 3: Cross-sections of the 3D model ─────────────────────────────────
# E-W cross-section at median northing
mid_y = dens_valid.Y.median()
margin = dens_valid.Y.std() * 0.5
ew_dens = dens_valid[(dens_valid.Y > mid_y - margin) & (dens_valid.Y < mid_y + margin)]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('3D Model Cross-Sections (E–W slice at median Northing)', fontsize=13, fontweight='bold')

sc3 = axes[0].scatter(ew_dens.X/1e3, ew_dens.Z/1e3, c=ew_dens.Value,
                      cmap='seismic', s=8, alpha=0.7, vmin=-0.3, vmax=-0.1)
plt.colorbar(sc3, ax=axes[0], label='Density contrast (g/cc)')
axes[0].set_title('Density — E–W Section', fontweight='bold')
axes[0].set_xlabel('Easting (km)'); axes[0].set_ylabel('Depth (km)')
axes[0].invert_yaxis()

# Depth profile: mean density vs depth
depth_bins = pd.cut(dens_valid.Z, bins=40)
depth_profile = dens_valid.groupby(depth_bins, observed=True)['Value'].agg(['mean','std'])
depth_profile.index = [iv.mid for iv in depth_profile.index]

axes[1].fill_betweenx(depth_profile.index/1e3,
                       depth_profile['mean'] - depth_profile['std'],
                       depth_profile['mean'] + depth_profile['std'],
                       alpha=0.25, color='royalblue', label='±1σ')
axes[1].plot(depth_profile['mean'], depth_profile.index/1e3, color='royalblue', lw=2, label='Mean')
axes[1].axvline(0, color='gray', lw=0.8, ls='--')
axes[1].set_xlabel('Mean density contrast (g/cc)')
axes[1].set_ylabel('Depth (km)')
axes[1].set_title('Density Depth Profile', fontweight='bold')
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3. Thermodynamic Database Exploration

The `DB0/` folder contains pre-computed thermodynamic equilibrium results stored as **Zarr** arrays. They encode how much H₂ (and other phases) is produced as a function of:
- **Pressure** (10–40,010 bar)
- **Temperature** (0–1450 °C)
- **Water-to-Rock ratio** (W/R = 0–0.2)

Two rock types are included:
- `HZ1` — Harzburgite
- `LH1` — Lherzolite

In [ ]:
def load_h2_db(rock_code, db_dir):
    """Load the H₂ production thermodynamic database for a given rock code."""
    path = os.path.join(db_dir, f'{rock_code}_all', f'{rock_code}_xarray_glpro_all.zarr')
    ds = xr.open_zarr(path)
    da = ds['__xarray_dataarray_variable__']
    return da

db_LH1 = load_h2_db('LH1', DB_DIR)
db_HZ1 = load_h2_db('HZ1', DB_DIR)

print('LH1 (Lherzolite) database:')
print(f'  Fields: {list(db_LH1.coords["fields"].values)}')
print(f'  Shape: {db_LH1.shape}')
print(f'  Pressure: {float(db_LH1.coords["pressure"].min()):.0f}–{float(db_LH1.coords["pressure"].max()):.0f} bar')
print(f'  Temperature: {int(db_LH1.coords["temperature"].min())}–{int(db_LH1.coords["temperature"].max())} °C')
print(f'  W/R ratios: {list(db_LH1.coords["w2r"].values)}')

In [ ]:
# ── Figure 4: H₂ Production vs Temperature (at different P and W/R) ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('H₂ Production from Thermodynamic Database (LH1 — Lherzolite)', fontsize=13, fontweight='bold')

pressures_mpa = [200, 500, 1000, 2000]   # MPa → bar = MPa * 10
T_range = slice(100, 700)
wr_fixed = 0.16

colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(pressures_mpa)))
for ax_idx, field in enumerate(['mol_kg_H2', 'mol_m3_H2']):
    for p_mpa, col in zip(pressures_mpa, colors):
        p_bar = p_mpa * 10
        arr = db_LH1.sel(pressure=p_bar, method='nearest', w2r=wr_fixed).sel(fields=field)
        arr_sel = arr.sel(temperature=slice(100, 700))
        axes[ax_idx].plot(arr_sel.coords['temperature'].values,
                          arr_sel.values, color=col, lw=2,
                          label=f'{p_mpa} MPa')

    axes[ax_idx].set_xlabel('Temperature (°C)')
    axes[ax_idx].legend(title='Pressure', fontsize=9)
    axes[ax_idx].axvline(300, color='gray', lw=0.8, ls='--', label='Optimal T ~300°C')

axes[0].set_ylabel('H₂ production (mol/kg rock)')
axes[0].set_title('H₂ per kg of Rock (W/R = 0.16)', fontweight='bold')
axes[1].set_ylabel('H₂ production (mol/m³)')
axes[1].set_title('H₂ per m³ (W/R = 0.16)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 5: H₂ Production Heatmap — T vs P ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('H₂ Production Heatmap (mol/kg rock, W/R = 0.16)', fontsize=13, fontweight='bold')

for ax, rock_name, db in zip(axes, ['LH1 (Lherzolite)', 'HZ1 (Harzburgite)'], [db_LH1, db_HZ1]):
    h2_map = db.sel(w2r=0.16, fields='mol_kg_H2').sel(
        temperature=slice(0, 800),
        pressure=slice(100, 20000)
    ).values  # shape: (pressure, temperature)

    T_vals = db.sel(temperature=slice(0,800)).coords['temperature'].values
    P_vals = db.sel(pressure=slice(100, 20000)).coords['pressure'].values / 10  # → MPa

    im = ax.pcolormesh(T_vals, P_vals, h2_map, cmap='YlOrRd', shading='auto')
    plt.colorbar(im, ax=ax, label='mol H₂ / kg rock')
    ax.set_xlabel('Temperature (°C)')
    ax.set_ylabel('Pressure (MPa)')
    ax.set_title(rock_name, fontweight='bold')
    # Mark serpentinization window
    ax.axvspan(200, 400, alpha=0.12, color='cyan', label='Optimal T window')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 6: Effect of Water-to-Rock Ratio ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
wr_values = [0.02, 0.06, 0.10, 0.14, 0.18, 0.20]
colors = plt.cm.viridis(np.linspace(0, 1, len(wr_values)))
P_fixed = 5000  # bar

for wr, col in zip(wr_values, colors):
    arr = db_LH1.sel(pressure=P_fixed, method='nearest', w2r=wr,
                     fields='mol_kg_H2').sel(temperature=slice(50, 600))
    ax.plot(arr.coords['temperature'].values, arr.values,
            color=col, lw=2, label=f'W/R = {wr}')

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('H₂ production (mol/kg rock)', fontsize=12)
ax.set_title('H₂ Production vs Temperature for Different W/R Ratios\n(LH1, P = 500 MPa)', fontweight='bold')
ax.legend(title='W/R ratio', ncol=2, fontsize=9)
ax.axvspan(200, 400, alpha=0.1, color='green', label='Optimal serpentinization window')
plt.tight_layout()
plt.show()

---
## 4. Core Physics

PoNHy implements three key physical relationships:

1. **Serpentinization degree** — inferred from the joint density/susceptibility inversion
2. **H₂ solubility** — via the Kumar & Killingley equation of state (KK-PR)
3. **Lithostatic pressure** — depth-integrated overburden

In [ ]:
# ── Serpentinization Degree lookup table ───────────────────────────────────────
# From ponhy.py: density-susceptibility pairs map to serpentinization (%)
SERP_DATA = {
    'x_points': np.array([2.62, 2.6355, 2.651, 2.6665, 2.682, 2.6975, 2.713, 2.7285,
                          2.744, 2.7595, 2.775, 2.7905, 2.806, 2.8215, 2.837, 2.8525,
                          2.868, 2.8835, 2.899, 2.9145, 2.93, 2.9455, 2.961, 2.9765,
                          2.992, 3.0075, 3.023, 3.0385, 3.054, 3.0695, 3.085, 3.1005,
                          3.116, 3.1315, 3.147, 3.1625, 3.178, 3.1935, 3.209, 3.2245, 3.24]),
    'y_points': np.array([0.07, 0.0682775, 0.066555, 0.0648325, 0.06311, 0.0613875,
                          0.059665, 0.0579425, 0.05622, 0.0544975, 0.052775, 0.0510525,
                          0.04933, 0.0476075, 0.045885, 0.0441625, 0.04244, 0.0407175,
                          0.038995, 0.0372725, 0.03555, 0.0338275, 0.032105, 0.0303825,
                          0.02866, 0.0269375, 0.025215, 0.0234925, 0.02177, 0.0200475,
                          0.018325, 0.0166025, 0.01488, 0.0131575, 0.011435, 0.0097125,
                          0.00799, 0.0062675, 0.004545, 0.0028225, 0.0011]),
    'values': np.arange(100, -2.5, -2.5),  # 100% → 0% in steps of 2.5%
}

def compute_serpentinization_degree(density, susceptibility):
    """Interpolate serpentinization degree (%) from density and susceptibility."""
    from scipy.interpolate import LinearNDInterpolator
    points = np.column_stack([SERP_DATA['x_points'], SERP_DATA['y_points']])
    interp = LinearNDInterpolator(points, SERP_DATA['values'])
    return interp(density, susceptibility)

# Visualise the lookup surface
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(SERP_DATA['x_points'], SERP_DATA['y_points'],
                c=SERP_DATA['values'], cmap='RdYlGn', s=80, zorder=3)
plt.colorbar(sc, ax=ax, label='Serpentinization degree (%)')
ax.set_xlabel('Density (g/cm³)', fontsize=12)
ax.set_ylabel('Magnetic susceptibility (SI)', fontsize=12)
ax.set_title('Serpentinization Degree Lookup\n(Density × Susceptibility → Serp%)', fontweight='bold')
# Draw arrows to show direction
ax.annotate('100% serpentinized\n(low ρ, high χ)', xy=(2.63, 0.068), fontsize=9, color='green',
            xytext=(2.67, 0.065), arrowprops=dict(arrowstyle='->', color='green'))
ax.annotate('0% serpentinized\n(high ρ, low χ)', xy=(3.23, 0.002), fontsize=9, color='red',
            xytext=(3.10, 0.015), arrowprops=dict(arrowstyle='->', color='red'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Apply serpentinization lookup to inversion data ───────────────────────────
# Merge density + magsus on common X,Y,Z
merged = dens_valid.rename(columns={'Value':'Density'}).merge(
    magsus_valid.rename(columns={'Value':'Susceptibility'}),
    on=['X','Y','Z'], how='inner'
)

# Convert density contrast to absolute density (background = 2.67 g/cc)
merged['Density_abs'] = 2.67 + merged['Density']

# Clip to valid lookup range
merged = merged[
    (merged.Density_abs >= SERP_DATA['x_points'].min()) &
    (merged.Density_abs <= SERP_DATA['x_points'].max()) &
    (merged.Susceptibility >= SERP_DATA['y_points'].min()) &
    (merged.Susceptibility <= SERP_DATA['y_points'].max())
]

merged['Serp_pct'] = compute_serpentinization_degree(
    merged.Density_abs.values, merged.Susceptibility.values
)

print(f'Cells with serpentinization estimate: {merged.Serp_pct.notna().sum():,}')
print(f'Mean serpentinization: {merged.Serp_pct.mean():.1f}%')
print(f'Max serpentinization: {merged.Serp_pct.max():.1f}%')

In [ ]:
# ── H₂ Solubility (Kumar-Killingley Peng-Robinson EOS) ───────────────────────
def compute_h2_solubility_kk_pr(p_mpa, t_c):
    """Compute H₂ solubility in water (mol/kg) using the KK-PR EOS.
    Adapted directly from PoNHy utils/general.py.
    """
    R = 83.14472       # cm³·bar/(mol·K)
    T_K = t_c + 273.15
    P_bar = p_mpa * 10.0

    # H₂ critical properties
    Tc = 33.19         # K
    Pc = 13.13         # bar
    omega = -0.216     # acentric factor

    Tr = T_K / Tc
    alpha = (1 + (0.37464 + 1.54226 * omega - 0.26992 * omega**2) * (1 - np.sqrt(Tr)))**2
    a = 0.45724 * (R * Tc)**2 / Pc * alpha
    b = 0.07780 * R * Tc / Pc

    A = a * P_bar / (R * T_K)**2
    B = b * P_bar / (R * T_K)

    # Cubic EOS coefficients
    coeffs = [1, -(1 - B), (A - 3*B**2 - 2*B), -(A*B - B**2 - B**3)]
    roots = np.roots(coeffs)
    real_roots = roots[np.isreal(roots)].real
    Z = max(real_roots[real_roots > 0]) if len(real_roots[real_roots > 0]) > 0 else 1.0

    # Fugacity coefficient
    ln_phi = Z - 1 - np.log(Z - B) - A / (2.828 * B) * np.log((Z + 2.414*B) / (Z - 0.414*B))
    fugacity = P_bar * np.exp(ln_phi)

    # KK correlation for solubility
    kH = np.exp(-4.7982 + 6.3536 / Tr + 9.0 * np.log(Tr))  # bar
    molality = fugacity / kH  # mol/kg
    return max(molality, 0.0)

# Generate solubility surface
T_grid = np.arange(50, 601, 10)   # °C
P_grid = np.arange(10, 501, 10)   # MPa
TT, PP = np.meshgrid(T_grid, P_grid)
sol_grid = np.vectorize(compute_h2_solubility_kk_pr)(PP, TT)

fig, ax = plt.subplots(figsize=(9, 5))
cf = ax.contourf(TT, PP, sol_grid, levels=20, cmap='Blues')
cs = ax.contour(TT, PP, sol_grid, levels=10, colors='navy', linewidths=0.5, alpha=0.4)
ax.clabel(cs, fmt='%.3f', fontsize=7)
plt.colorbar(cf, ax=ax, label='H₂ solubility (mol/kg H₂O)')
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Pressure (MPa)', fontsize=12)
ax.set_title('H₂ Solubility in Water\n(Kumar-Killingley Peng-Robinson EOS)', fontweight='bold')
ax.axvspan(200, 400, alpha=0.1, color='orange', label='Optimal T window')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Lithostatic Pressure Profile ───────────────────────────────────────────────
def compute_lithostatic_pressure(depth_m, rho_kg_m3=2700.0, g=9.81):
    """Compute lithostatic pressure at depth. Returns pressure in MPa."""
    return (rho_kg_m3 * g * depth_m) / 1e6

depths_km = np.linspace(0, 20, 200)
P_litho = compute_lithostatic_pressure(depths_km * 1e3)

fig, ax = plt.subplots(figsize=(5, 7))
ax.plot(P_litho, depths_km, color='steelblue', lw=2.5)
ax.fill_betweenx(depths_km, P_litho, alpha=0.15, color='steelblue')
ax.set_xlabel('Lithostatic Pressure (MPa)', fontsize=12)
ax.set_ylabel('Depth (km)', fontsize=12)
ax.invert_yaxis()
ax.set_title('Lithostatic Pressure Profile\n(ρ = 2700 kg/m³, g = 9.81 m/s²)', fontweight='bold')
ax.grid(True, ls='--', alpha=0.3)
# Mark serpentinization depth window
ax.axhspan(3, 12, alpha=0.1, color='green', label='Serpentinization zone (3–12 km)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Production Rate Calculation

PoNHy estimates H₂ production by:
1. Binning serpentinite cells by temperature range
2. Looking up H₂ rate (mol/kg) from the thermodynamic DB at local P, T, W/R
3. Applying the serpentinization correction factor
4. Scaling by rock mass and simulation duration

In [ ]:
# ── Merge temperature data with serpentinization model ─────────────────────────
# Bin the temperature into ranges used by PoNHy
TEMP_RANGES = [(100,150), (150,200), (200,250), (250,300), (300,350),
               (350,400), (400,450), (450,500)]

# Sample rates from the thermodynamic DB at each temperature bin center
WR = 0.16
DENSITY_LITHO = 2700  # kg/m³
GRAVITY = 9.81

production_summary = []
for T_lo, T_hi in TEMP_RANGES:
    T_mid = (T_lo + T_hi) / 2
    P_mpa = compute_lithostatic_pressure(8000)  # assume 8 km depth for this range
    P_bar = P_mpa * 10

    # Query DB
    h2_rate = float(db_LH1.sel(
        pressure=P_bar, method='nearest',
        temperature=T_mid, method='nearest',
        w2r=WR
    ).sel(fields='mol_kg_H2').values)

    solubility = compute_h2_solubility_kk_pr(P_mpa, T_mid)

    production_summary.append({
        'T_range': f'{T_lo}_{T_hi}',
        'T_mid': T_mid,
        'P_MPa': P_mpa,
        'mol_kg_H2': h2_rate,
        'kg_H2_per_t_rock': h2_rate * 0.002016 * 1e3,  # mol/kg → g/kg → kg/t
        'H2_solubility_mol_kg': solubility,
    })

prod_df = pd.DataFrame(production_summary)
print(prod_df.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ── Figure 7: Production rate summary by temperature range ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('H₂ Production Estimates by Temperature Range (LH1, W/R=0.16)', fontsize=13, fontweight='bold')

x = prod_df['T_range']
axes[0].bar(x, prod_df['mol_kg_H2'], color=plt.cm.YlOrRd(np.linspace(0.3,0.9,len(x))))
axes[0].set_xlabel('Temperature range (°C)')
axes[0].set_ylabel('H₂ production rate (mol/kg rock)')
axes[0].set_title('Volumetric H₂ Rate')
axes[0].tick_params(axis='x', rotation=35)

axes[1].bar(x, prod_df['H2_solubility_mol_kg'], color=plt.cm.Blues(np.linspace(0.4,0.9,len(x))))
axes[1].set_xlabel('Temperature range (°C)')
axes[1].set_ylabel('H₂ solubility (mol/kg H₂O)')
axes[1].set_title('H₂ Solubility (KK-PR EOS)')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

---
## 6. Monte Carlo Sensitivity Analysis

PoNHy quantifies uncertainty in H₂ estimates via Monte Carlo sampling over key parameters. Here we implement a lightweight version of the **no-saturation** workflow.

In [ ]:
np.random.seed(42)

N_ITER = 5000

# Base parameter values (from Pyrenees config)
BASE = {
    'prod_rate':      prod_df.loc[prod_df.T_mid==325, 'mol_kg_H2'].values[0],  # mol/kg at ~325°C
    'serp_volume_km3': 2.5,      # km³ of serpentinite (example)
    'serp_degree_pct': 45.0,     # mean serpentinization %
    'v_ref':          1.272e-5,  # cm/day serpentinization front velocity
    'years':          1.0,
    'density_serp':   2910.0,    # kg/m³
}

# Parameter uncertainty ranges (multiplicative factors)
RANGES = {
    'prod_rate':       (0.5, 1.5),
    'serp_volume_km3': (0.8, 1.2),
    'serp_degree_pct': (0.8, 1.2),
    'v_ref':           (0.4, 1.6),
}

MOLAR_MASS_H2 = 0.00201588  # kg/mol

h2_tons_mc = []
for _ in range(N_ITER):
    # Sample multiplicative factors
    f_prod    = np.random.uniform(*RANGES['prod_rate'])
    f_vol     = np.random.uniform(*RANGES['serp_volume_km3'])
    f_serp    = np.random.uniform(*RANGES['serp_degree_pct'])
    f_v       = np.random.uniform(*RANGES['v_ref'])

    prod_rate  = BASE['prod_rate'] * f_prod          # mol/kg rock
    volume_m3  = BASE['serp_volume_km3'] * f_vol * 1e9  # m³
    serp_deg   = BASE['serp_degree_pct'] * f_serp / 100.0  # fraction

    # Rock mass in serpentinite volume
    mass_rock_kg = volume_m3 * BASE['density_serp'] * serp_deg

    # Annual H₂ production
    mol_h2 = prod_rate * mass_rock_kg
    kg_h2  = mol_h2 * MOLAR_MASS_H2
    tons_h2 = kg_h2 / 1e3
    h2_tons_mc.append(tons_h2)

h2_tons_mc = np.array(h2_tons_mc)

print(f'Monte Carlo H₂ production (N={N_ITER:,} iterations):')
print(f'  Mean:   {np.mean(h2_tons_mc):,.0f} tons/yr')
print(f'  Median: {np.median(h2_tons_mc):,.0f} tons/yr')
print(f'  P5–P95: {np.percentile(h2_tons_mc,5):,.0f} – {np.percentile(h2_tons_mc,95):,.0f} tons/yr')

In [ ]:
# ── Figure 8: MC Distribution ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Monte Carlo H₂ Production Distribution (No-Saturation)', fontsize=13, fontweight='bold')

# Histogram
axes[0].hist(h2_tons_mc, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(np.mean(h2_tons_mc), color='tomato', lw=2, ls='--', label=f'Mean: {np.mean(h2_tons_mc):,.0f} t/yr')
axes[0].axvline(np.percentile(h2_tons_mc, 5),  color='orange', lw=1.5, ls=':', label='P5/P95')
axes[0].axvline(np.percentile(h2_tons_mc, 95), color='orange', lw=1.5, ls=':')
axes[0].set_xlabel('H₂ production (tons/yr)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution', fontweight='bold')
axes[0].legend(fontsize=9)

# KDE + CDF
kde = gaussian_kde(h2_tons_mc)
x_kde = np.linspace(h2_tons_mc.min(), h2_tons_mc.max(), 300)
sorted_mc = np.sort(h2_tons_mc)
cdf = np.arange(1, N_ITER+1) / N_ITER

ax2 = axes[1].twinx()
axes[1].plot(x_kde, kde(x_kde), color='steelblue', lw=2.5, label='KDE')
ax2.plot(sorted_mc, cdf, color='tomato', lw=2, label='CDF')
axes[1].set_xlabel('H₂ production (tons/yr)', fontsize=12)
axes[1].set_ylabel('Probability density', fontsize=12, color='steelblue')
ax2.set_ylabel('Cumulative probability', fontsize=12, color='tomato')
axes[1].set_title('KDE & CDF', fontweight='bold')
lines1, lab1 = axes[1].get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1+lines2, lab1+lab2, fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 9: Sensitivity — Tornado chart ─────────────────────────────────────
# Run one-at-a-time sensitivity
def h2_from_params(f_prod=1, f_vol=1, f_serp=1, f_v=1):
    prod_rate  = BASE['prod_rate'] * f_prod
    volume_m3  = BASE['serp_volume_km3'] * f_vol * 1e9
    serp_deg   = BASE['serp_degree_pct'] * f_serp / 100.0
    mass_rock_kg = volume_m3 * BASE['density_serp'] * serp_deg
    mol_h2 = prod_rate * mass_rock_kg
    return mol_h2 * MOLAR_MASS_H2 / 1e3

baseline = h2_from_params()
params = ['Production rate', 'Serp. volume', 'Serp. degree', 'Front velocity']
lows, highs = [], []
for fname, rng in RANGES.items():
    kwargs_lo = {f'f_{k}': 1.0 for k in RANGES}
    kwargs_hi = {f'f_{k}': 1.0 for k in RANGES}
    key = f'f_{fname}'
    kwargs_lo[key] = rng[0]
    kwargs_hi[key] = rng[1]
    lows.append(h2_from_params(**kwargs_lo) - baseline)
    highs.append(h2_from_params(**kwargs_hi) - baseline)

fig, ax = plt.subplots(figsize=(9, 4))
y_pos = np.arange(len(params))
ax.barh(y_pos, highs, left=0, color='steelblue', label='High (+factor)', alpha=0.85)
ax.barh(y_pos, lows, left=0, color='tomato', label='Low (−factor)', alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(params, fontsize=11)
ax.set_xlabel('Change in H₂ output from baseline (tons/yr)', fontsize=11)
ax.set_title('Sensitivity Analysis — Tornado Chart\n(Impact of each parameter at its min/max)', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 7. Full Visualization Suite

In [ ]:
# ── Figure 10: Serpentinization Heatmap ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Serpentinization Degree — Map View', fontsize=14, fontweight='bold')

# Map: shallowest layer
shallow_layer = merged[merged.Z == merged.Z.max()]
sc = axes[0].scatter(shallow_layer.X/1e3, shallow_layer.Y/1e3,
                     c=shallow_layer.Serp_pct, cmap='RdYlGn',
                     s=20, vmin=0, vmax=100)
plt.colorbar(sc, ax=axes[0], label='Serpentinization (%)')
axes[0].set_xlabel('Easting (km)'); axes[0].set_ylabel('Northing (km)')
axes[0].set_title(f'Shallow slice (Z ≈ {merged.Z.max():.0f} m)', fontweight='bold')

# Depth profile
depth_serp = merged.groupby(pd.cut(merged.Z, bins=30), observed=True)['Serp_pct'].agg(['mean','std'])
depth_serp.index = [iv.mid for iv in depth_serp.index]
axes[1].fill_betweenx(depth_serp.index/1e3,
                       depth_serp['mean'] - depth_serp['std'],
                       depth_serp['mean'] + depth_serp['std'],
                       alpha=0.2, color='green')
axes[1].plot(depth_serp['mean'], depth_serp.index/1e3, color='green', lw=2)
axes[1].set_xlabel('Mean serpentinization degree (%)', fontsize=11)
axes[1].set_ylabel('Depth (km)', fontsize=11)
axes[1].set_title('Serpentinization vs Depth', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 11: Temperature Field Map ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('3D Temperature Field — Pyrenees Case Study', fontsize=13, fontweight='bold')

# Shallow temperature map
t_shallow = temp_df[temp_df.Z > temp_df.Z.quantile(0.95)]
sc = axes[0].scatter(t_shallow.X/1e3, t_shallow.Y/1e3,
                     c=t_shallow.Temperature, cmap='hot_r', s=4)
plt.colorbar(sc, ax=axes[0], label='Temperature (°C)')
axes[0].set_title('Near-surface temperature', fontweight='bold')
axes[0].set_xlabel('Easting (km)'); axes[0].set_ylabel('Northing (km)')

# Geothermal gradient
t_profile = temp_df.groupby(pd.cut(temp_df.Z, bins=50), observed=True)['Temperature'].mean()
t_profile.index = [iv.mid for iv in t_profile.index]
axes[1].plot(t_profile.values, t_profile.index/1e3, color='crimson', lw=2)
axes[1].set_xlabel('Temperature (°C)', fontsize=11)
axes[1].set_ylabel('Elevation / Depth (km)', fontsize=11)
axes[1].set_title('Geothermal Profile', fontweight='bold')
axes[1].invert_yaxis()
axes[1].axhspan(3, 12, alpha=0.1, color='orange', label='Serpentinization zone')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 8. Interactive Parameter Explorer

Use the sliders below to explore how input parameters affect the H₂ production estimate.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def compute_h2_estimate(temp_c, pressure_mpa, wr, serp_pct, volume_km3, density_serp):
    """Quick H₂ estimate from user-controlled parameters."""
    # DB lookup
    P_bar = pressure_mpa * 10
    h2_mol_kg = float(db_LH1.sel(
        pressure=P_bar, method='nearest',
        temperature=temp_c, method='nearest',
        w2r=wr, method='nearest'
    ).sel(fields='mol_kg_H2').values)

    solubility = compute_h2_solubility_kk_pr(pressure_mpa, temp_c)

    volume_m3  = volume_km3 * 1e9
    serp_frac  = serp_pct / 100.0
    mass_kg    = volume_m3 * density_serp * serp_frac
    mol_h2     = h2_mol_kg * mass_kg
    tons_h2    = mol_h2 * MOLAR_MASS_H2 / 1e3

    return h2_mol_kg, solubility, tons_h2

# Widgets
w_temp    = widgets.FloatSlider(value=300, min=50, max=600, step=10, description='T (°C)', style={'description_width':'120px'})
w_press   = widgets.FloatSlider(value=500, min=10, max=3000, step=10, description='P (MPa)', style={'description_width':'120px'})
w_wr      = widgets.SelectionSlider(options=[0.0,0.02,0.04,0.06,0.08,0.10,0.12,0.14,0.16,0.18,0.20],
                                     value=0.16, description='W/R ratio', style={'description_width':'120px'})
w_serp    = widgets.FloatSlider(value=45, min=5, max=100, step=5, description='Serp. degree (%)', style={'description_width':'120px'})
w_vol     = widgets.FloatSlider(value=2.5, min=0.1, max=20, step=0.1, description='Volume (km³)', style={'description_width':'120px'})
w_density = widgets.FloatSlider(value=2910, min=2500, max=3200, step=10, description='ρ serp (kg/m³)', style={'description_width':'120px'})
output    = widgets.Output()

def update(*args):
    with output:
        output.clear_output(wait=True)
        h2_rate, solub, tons = compute_h2_estimate(
            w_temp.value, w_press.value, w_wr.value,
            w_serp.value, w_vol.value, w_density.value
        )

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle('PoNHy Interactive H₂ Estimator', fontsize=13, fontweight='bold')

        # H₂ production bar
        axes[0].bar(['H₂ production'], [h2_rate], color='steelblue')
        axes[0].set_ylabel('mol/kg rock')
        axes[0].set_title(f'Thermo rate: {h2_rate:.4f} mol/kg', fontweight='bold')

        # Solubility indicator
        axes[1].bar(['H₂ solubility'], [solub], color='seagreen')
        axes[1].set_ylabel('mol/kg H₂O')
        axes[1].set_title(f'Solubility: {solub:.4f} mol/kg', fontweight='bold')

        # Total H₂
        color = 'gold' if tons < 1e4 else ('orange' if tons < 1e6 else 'tomato')
        axes[2].bar(['Total H₂/yr'], [tons/1e3], color=color)
        axes[2].set_ylabel('Kilotons H₂/yr')
        axes[2].set_title(f'H₂ output: {tons:,.0f} tons/yr', fontweight='bold')

        plt.tight_layout()
        plt.show()

for w in [w_temp, w_press, w_wr, w_serp, w_vol, w_density]:
    w.observe(update, names='value')

ui = widgets.VBox([
    widgets.HTML('<h3>🔬 PoNHy Parameter Explorer</h3>'),
    widgets.HBox([widgets.VBox([w_temp, w_press, w_wr]),
                  widgets.VBox([w_serp, w_vol, w_density])]),
    output
])
display(ui)
update()

---
## Summary

| Step | What we did |
|------|-------------|
| Data unpacking | Loaded topography, gravity, magnetics, density, susceptibility, and temperature fields |
| DB exploration | Inspected the 4D Zarr thermodynamic database (T × P × W/R → H₂ fields) |
| Core physics | Implemented serpentinization degree lookup, KK-PR solubility, and lithostatic pressure |
| Production rates | Estimated H₂ output by temperature range using DB lookups |
| Monte Carlo | Ran N=5,000 uncertainty propagation simulations |
| Visualization | Maps, cross-sections, heatmaps, distributions, tornado chart |
| Interactive | ipywidgets dashboard for real-time parameter exploration |

To run the full inversion + H₂ quantification pipeline, use:
```bash
conda activate hydrogen
python ponhy.py
```

See the [README](https://github.com/RodolfoChristiansen/PoNHy) for configuration details.
